In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

A string value was discovered in the rating column of the ratings.csv file while building the model; as a result we are cleaning the csv file:

In [ ]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
links = pd.read_csv('links.csv')
tags = pd.read_csv('tags.csv')

In [ ]:
#because of irregular data: with conversion to numeric all non convertibles will be replaced with NaN values
ratings['rating'] = pd.to_numeric(ratings['rating'], errors='coerce')

#here we can drop the rows that contain NaN values in the rating column
ratings = ratings.dropna(subset=['rating'])

#the changes are then read back to our dataset csv file and ratings is read again
ratings.to_csv('ratings.csv', index=False)
ratings = pd.read_csv('ratings.csv')

early rounds of analysing the data;

In [ ]:
movies.head(1)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


In [ ]:
ratings.head(1)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703


In [ ]:
links.head(1)

,movieId,imdbId,tmdbId
0,1,114709,862.0


In [ ]:
ratings.head(1)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703


# EXPLORATORY DATA ANALYSIS:

# Checking for dimensions:

In [ ]:
print(movies.shape)
print(links.shape)
print(ratings.shape)
print(tags.shape)


(9742, 3)
(9742, 3)
(100836, 4)
(3683, 4)


# Checking to see if any of the values are null:

In [ ]:
links.isnull().sum()

movieId    0
imdbId     0
tmdbId     8
dtype: int64

In [ ]:
tags.isnull().sum()

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

In [ ]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [ ]:
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

# Checking to see if any of our values are duplicated:

In [ ]:
movies.duplicated().sum()

0

In [ ]:
ratings.duplicated().sum()

0

In [ ]:
links.duplicated().sum()

0

In [ ]:
tags.duplicated().sum()

0

# Splitting the ratings data into training and testing data sets

In [ ]:
# Split ratings data into training and testing sets
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

Firstly building a popularity based recommendation system

Creating a dataframe that has the movie details and its corresponding ratings

In [ ]:
ratingsAndMovies = ratings.merge(movies,on='movieId')

We only wish to consider the ratings of the books while checking for popularity that have been rated by more than 200 users; now lets count the number of users that have rated the movies

In [ ]:
countRatingsDf = ratingsAndMovies.groupby('title').count()['rating'].reset_index()
countRatingsDf.rename(columns={'rating': 'numberOfRatings'}, inplace=True)
countRatingsDf

,title,numberOfRatings
0,'71 (2014),1
1,'Hellboy': The Seeds of Creation (2004),1
2,'Round Midnight (1986),2
3,'Salem's Lot (2004),1
4,'Til There Was You (1997),2
...,...,...
9714,eXistenZ (1999),22
9715,xXx (2002),24
9716,xXx: State of the Union (2005),5
9717,¡Three Amigos! (1986),26


After calculating the number of ratings we need to analyse the average rating of each movie:

In [ ]:
# Now, calculate the mean of 'rating' grouped by 'title'
averageRatingsDf = ratingsAndMovies.groupby('title')['rating'].mean().reset_index()
averageRatingsDf.rename(columns={'rating': 'averageOfRatings'}, inplace=True)

# Print the resulting DataFrame
print(averageRatingsDf)


                                          title  averageOfRatings
0                                    '71 (2014)          4.000000
1       'Hellboy': The Seeds of Creation (2004)          4.000000
2                        'Round Midnight (1986)          3.500000
3                           'Salem's Lot (2004)          5.000000
4                     'Til There Was You (1997)          4.000000
...                                         ...               ...
9714                            eXistenZ (1999)          3.863636
9715                                 xXx (2002)          2.770833
9716             xXx: State of the Union (2005)          2.000000
9717                      ¡Three Amigos! (1986)          3.134615
9718  À nous la liberté (Freedom for Us) (1931)          1.000000

[9719 rows x 2 columns]


In [ ]:
popularityDf = countRatingsDf.merge(averageRatingsDf)
popularityDf

,title,numberOfRatings,averageOfRatings
0,'71 (2014),1,4.000000
1,'Hellboy': The Seeds of Creation (2004),1,4.000000
2,'Round Midnight (1986),2,3.500000
3,'Salem's Lot (2004),1,5.000000
4,'Til There Was You (1997),2,4.000000
...,...,...,...
9714,eXistenZ (1999),22,3.863636
9715,xXx (2002),24,2.770833
9716,xXx: State of the Union (2005),5,2.000000
9717,¡Three Amigos! (1986),26,3.134615


now we only need to check the books that have been rated by more than 200 users so:

In [ ]:

popularityDf = popularityDf[popularityDf['numberOfRatings'] >= 200]

# Lets sort such that the movies that have been rated the highest are on top:
popularityDf = popularityDf.sort_values('averageOfRatings', ascending=False)

Our popularity dataframe now has 19 of the most popular movies.

In [ ]:
popularityDf.shape

(19, 3)

The popularity dataframe should consist of some details like the genres of the movie. So we will merge our popularity dataframe with movies.

In [ ]:
popularityDf.merge(movies, on='title').drop_duplicates('title')

We can also add the tags of the movie that should be displayed in Top 10

In [ ]:
#popularityDf.merge(tags, on='movieId').drop_duplicates('title')
#fix this error in next version

In [ ]:
print("Top 10: ")
popularityDf.head(10)

Top 10: 


,title,numberOfRatings,averageOfRatings
7593,"Shawshank Redemption, The (1994)",317,4.429022
3011,Fight Club (1999),218,4.272936
9119,"Usual Suspects, The (1995)",204,4.237745
8001,Star Wars: Episode IV - A New Hope (1977),251,4.231076
7421,Schindler's List (1993),220,4.225000
8002,Star Wars: Episode V - The Empire Strikes Back...,211,4.215640
6944,Raiders of the Lost Ark (Indiana Jones and the...,200,4.207500
6865,Pulp Fiction (1994),307,4.197068
5512,"Matrix, The (1999)",278,4.192446
3158,Forrest Gump (1994),329,4.164134


# Collaborative Filtering Based Recommendation System

To make the system more intelligent we should only consider the rows such that the users whose reviews are being considered have rated at least 100 movies and the movies that are being considered have at least been rated by 50 users

We need to make a table where the row names are the movie names and the column names are the user and R X C is the rating that C has given to R. This is the foundation of our recommendation system using collaborative filtering.

let us check how many ratings have been given by each user and filter the data to 100 ratings minimum.

In [ ]:
x = ratingsAndMovies.groupby('userId').count()['rating'] > 100
#this returns true for the users that have and false for the ones that havent
filmGeeks = x[x].index
#filmGeeks will only store the users that have a rating > 100

Now we will filter our dataframe using isin() such that only those rows are returned where the users are one of the filmGeeks.

In [ ]:
userFilteredRatings = ratingsAndMovies[ratingsAndMovies['userId'].isin(filmGeeks)]

In [ ]:
y = userFilteredRatings.groupby('title').count()['rating'] >= 50
#this returns a boolean set of values that we can store in y
# y[y] this stores only the boolean true returning values of y in y
highRatedMovies = y[y].index
highRatedMovies

Index(['2001: A Space Odyssey (1968)', '300 (2007)',
       '40-Year-Old Virgin, The (2005)', 'A.I. Artificial Intelligence (2001)',
       'Abyss, The (1989)', 'Ace Ventura: Pet Detective (1994)',
       'Ace Ventura: When Nature Calls (1995)', 'Air Force One (1997)',
       'Airplane! (1980)', 'Aladdin (1992)',
       ...
       'When Harry Met Sally... (1989)', 'While You Were Sleeping (1995)',
       'Who Framed Roger Rabbit? (1988)',
       'Willy Wonka & the Chocolate Factory (1971)',
       'Wizard of Oz, The (1939)', 'X-Men (2000)',
       'X-Men: The Last Stand (2006)', 'X2: X-Men United (2003)',
       'Young Frankenstein (1974)', 'Zoolander (2001)'],
      dtype='object', name='title', length=331)

In [ ]:
userFilteredRatings['title'].isin(highRatedMovies)
#this will return boolean values
finalRatings = userFilteredRatings[userFilteredRatings['title'].isin(highRatedMovies)]
finalRatings

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,17,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
5,18,1,3.5,1455209816,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
...,...,...,...,...,...,...
74478,594,1393,5.0,1109037039,Jerry Maguire (1996),Drama|Romance
74479,599,1393,4.0,1498499718,Jerry Maguire (1996),Drama|Romance
74480,600,1393,3.5,1237850723,Jerry Maguire (1996),Drama|Romance
74481,606,1393,3.5,1228164987,Jerry Maguire (1996),Drama|Romance


we use a pivot_table() pandas function here to create a pivot table which is a spreadsheet like table where we can set the index, columns and row attributes

In [ ]:
pt = finalRatings.pivot_table(index = 'title', columns = 'userId', values = 'rating')
pt.fillna(0,inplace=True)
pt
# since 331 rows that is 331 movies are returned we need to calculate the euclidean distance of
#each movie with each other movie as vectors

userId,1,4,6,7,10,15,17,18,19,20,...,599,600,601,602,603,605,606,607,608,610
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,3.0,0.0,...,5.0,4.0,0.0,0.0,5.0,0.0,5.0,0.0,3.0,4.5
300 (2007),0.0,0.0,0.0,0.0,3.0,0.0,0.0,3.5,0.0,0.0,...,3.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,5.0,4.0
"40-Year-Old Virgin, The (2005)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.5,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.5
A.I. Artificial Intelligence (2001),0.0,0.0,0.0,4.5,0.0,4.0,0.0,0.0,0.0,3.0,...,2.5,3.5,0.0,0.0,0.0,1.0,3.5,0.0,4.5,3.5
"Abyss, The (1989)",4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,3.5,3.5,0.0,0.0,1.0,0.0,0.0,0.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
X-Men (2000),5.0,0.0,0.0,3.5,0.0,0.0,0.0,4.0,4.0,3.5,...,3.5,3.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,3.5
X-Men: The Last Stand (2006),0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,...,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,3.0
X2: X-Men United (2003),0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,0.0,4.0,...,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0


In [ ]:
similarityScores = cosine_similarity(pt)
# cosine_similarity(pt) returns 331 x 331 values

In [ ]:
finalRatings.drop_duplicates('title')

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
267,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
369,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
572,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
854,1,110,4.0,964982176,Braveheart (1995),Action|Drama|War
...,...,...,...,...,...,...
71144,27,1387,2.0,962686343,Jaws (1975),Action|Horror
72747,28,8874,2.0,1243837559,Shaun of the Dead (2004),Comedy|Horror
73298,28,48385,4.0,1242290334,Borat: Cultural Learnings of America for Make ...,Comedy
73483,28,51255,1.0,1234338068,Hot Fuzz (2007),Action|Comedy|Crime|Mystery


In [ ]:
similarityScores.shape

(331, 331)

In [ ]:
def recommend(bookname):
    if bookname in pt.index:
        index = np.where(pt.index == bookname)[0][0]
        similarItems = sorted(list(enumerate(similarityScores[index])), key=lambda x: x[1], reverse=True)[1:6]
        for i in similarItems:
            print(pt.index[i[0]])
    else:
        print(f"{bookname} is not included in our high priority dataset for collaborative filtering.")


In [ ]:
checkValue = ratingsAndMovies[ratingsAndMovies['title'] == 'Saving Private Ryan (1998)'].head(1)
print(checkValue)

       userId  movieId  rating  timestamp                       title  \
10778       1     2028     4.0  964981888  Saving Private Ryan (1998)   

                 genres  
10778  Action|Drama|War  


In [ ]:
recommend('Braveheart (1995)')

Terminator 2: Judgment Day (1991)
Forrest Gump (1994)
Saving Private Ryan (1998)
Jurassic Park (1993)
Pulp Fiction (1994)


In [ ]:
recommend('Jumanji')

Jumanji is not included in our high priority dataset for collaborative filtering.


In [ ]:
recommend('Jerry Maguire (1996)')

When Harry Met Sally... (1989)
Pretty Woman (1990)
Star Wars: Episode VI - Return of the Jedi (1983)
Rain Man (1988)
Shakespeare in Love (1998)


In [ ]:
recommend('Shaun of the Dead (2004)')

Hot Fuzz (2007)
Kill Bill: Vol. 2 (2004)
Kill Bill: Vol. 1 (2003)
Sin City (2005)
Donnie Darko (2001)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import numpy as np


movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
links = pd.read_csv('links.csv')

# to connect to TDMB
movies = movies.merge(links, on='movieId')

# splitting data except one rating per user for collaborative filtering
def split_data_except_one(df):
    test_data = pd.DataFrame(columns=df.columns)
    for user_id, group in df.groupby('userId'):
        if len(group) > 1:
            test_data = pd.concat([test_data, group.tail(1)])
            df = df.drop(group.tail(1).index)
    train_data = df
    return train_data, test_data

train_data, test_data = split_data_except_one(ratings)

# pivot table
pivot_data = train_data.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

# calculating the similarity between the users
user_similarity = cosine_similarity(pivot_data)

# user profile information
def get_user_profile(user_id):
    user_movies = pivot_data.loc[user_id]
    preferred_genres = set()
    for movie_id in user_movies.index:
        if user_movies[movie_id] > 0:
            movie_genres = movies[movies['movieId'] == movie_id]['genres'].values[0]
            preferred_genres.update(movie_genres.split('|'))
    return preferred_genres

# function to get movie details
def get_movie_details(movie_id):
    movie_details = movies[movies['movieId'] == movie_id]
    if not movie_details.empty:
        return {
            'movieId': movie_id,
            'title': movie_details['title'].values[0],
            'genres': movie_details['genres'].values[0]
        }
    else:
        return None

# function to get user recommendations with details and user profile information
def get_user_recommendations_with_profile(user_id, n=10):
    # get user profile information (preferred genres)
    user_profile = get_user_profile(user_id)
    if not user_profile:
        print("User has not provided any ratings.")
        return []

    # find similar users
    similar_users = user_similarity[user_id - 1].argsort()[::-1][1:n+1]

    # get movies rated by similar users but not by the target user
    recommended_movies = []
    for similar_user in similar_users:
        rated_movies = pivot_data.iloc[similar_user][pivot_data.iloc[similar_user] > 0].index.tolist()
        target_user_rated_movies = pivot_data.iloc[user_id - 1][pivot_data.iloc[user_id - 1] > 0].index.tolist()
        for movie_id in rated_movies:
            if movie_id not in target_user_rated_movies:
                movie_details = get_movie_details(movie_id)
                if movie_details:
                    movie_details['match_score'] = len(user_profile.intersection(set(movie_details['genres'].split('|'))))
                    recommended_movies.append(movie_details)
                    if len(recommended_movies) >= n:
                        break
        if len(recommended_movies) >= n:
            break

    return recommended_movies

# function to calculate predicted rating based on match score and average rating of the movie
def calculate_predicted_rating(match_score, average_rating):
    # match score can be scaled to be between 0 and 1
    match_score_scaled = match_score / 5  #  match_score can range from 0 to 5
    # predicted rating
    predicted_rating = (match_score_scaled * 5 + average_rating) / 2
    return predicted_rating

# ask user to choose a movie and rate it
def choose_movie_to_rate(recommended_movies):
    print("Choose a movie to rate (Enter the number corresponding to the movie):")
    for i, movie in enumerate(recommended_movies, start=1):
        print(f"{i}. Title: {movie['title']}")
        print(f"   Genres: {movie['genres']}")
    choice = int(input("Enter your choice (1-5): "))
    return recommended_movies[choice - 1]

# average ratings for movies
ratingsAndMovies = ratings.merge(movies, on='movieId')
averageRatingsDf = ratingsAndMovies.groupby('title')['rating'].mean().reset_index()
averageRatingsDf.rename(columns={'rating': 'averageOfRatings'}, inplace=True)

#calculate average rating of a movie based on ratings from similar users
def calculate_average_rating(movie_title, similar_users):
    avg_rating = 0
    count = 0
    for user in similar_users:
        user_rating = ratingsAndMovies[(ratingsAndMovies['userId'] == user) & (ratingsAndMovies['title'] == movie_title)]['rating'].values
        if len(user_rating) > 0:
            avg_rating += user_rating[0]
            count += 1
    if count > 0:
        avg_rating /= count
    return avg_rating


user_id = int(input("Enter the userID:"))
user_profile = get_user_profile(user_id)
print("User ID:", user_id)
print("Genres Watched:", ", ".join(user_profile))

recommended_movies = get_user_recommendations_with_profile(user_id, n=5)

# print recommended movies with details
print("\nRecommended Movies:")
for i, movie in enumerate(recommended_movies, start=1):
    print(f"{i}. Title: {movie['title']}")
    print(f"   Genres: {movie['genres']}")
    print(f"   Match Score (Genre Match with User Profile): {movie['match_score']}")
    print("-" * 30)

# choose a movie and rate it
chosen_movie = choose_movie_to_rate(recommended_movies)

# similar users
similar_users = user_similarity[user_id - 1].argsort()[::-1][1:6]  # Assuming 5 similar users

# average rating of the chosen movie based on ratings from similar users
average_rating = calculate_average_rating(chosen_movie['title'], similar_users)

actual_rating = float(input(f"Please rate '{chosen_movie['title']}' (1-5): "))
print(f"Actual rating for '{chosen_movie['title']}': {actual_rating}")

# predicted rating
predicted_rating = round(calculate_predicted_rating(chosen_movie['match_score'], average_rating),2)
print(f"Predicted rating for '{chosen_movie['title']}': {predicted_rating}")

# RMSE
rmse = round(np.sqrt(mean_squared_error([predicted_rating], [actual_rating])),2)
print("Root Mean Square Error (RMSE):", rmse)

# accuracy in terms of percentage
accuracy = round((1 - (rmse / 5)) * 100,2)  # Maximum rating is 5
print("Accuracy:", accuracy, "%")


Enter the userID:4
User ID: 4
Genres Watched: Documentary, Mystery, IMAX, Musical, Action, Western, Film-Noir, Fantasy, Comedy, Children, Animation, Romance, Horror, Adventure, Drama, Crime, Thriller, War, Sci-Fi

Recommended Movies:
1. Title: Toy Story (1995)
   Genres: Adventure|Animation|Children|Comedy|Fantasy
   Match Score (Genre Match with User Profile): 5
------------------------------
2. Title: Heat (1995)
   Genres: Action|Crime|Thriller
   Match Score (Genre Match with User Profile): 3
------------------------------
3. Title: GoldenEye (1995)
   Genres: Action|Adventure|Thriller
   Match Score (Genre Match with User Profile): 3
------------------------------
4. Title: Leaving Las Vegas (1995)
   Genres: Drama|Romance
   Match Score (Genre Match with User Profile): 2
------------------------------
5. Title: City of Lost Children, The (Cité des enfants perdus, La) (1995)
   Genres: Adventure|Drama|Fantasy|Mystery|Sci-Fi
   Match Score (Genre Match with User Profile): 5
-------

In [ ]:
checkValue = ratingsAndMovies[ratingsAndMovies['title'] == 'Saving Private Ryan (1998)'].head(1)
checkValue

,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId
10778,1,2028,4.0,964981888,Saving Private Ryan (1998),Action|Drama|War,120815,857.0


In [ ]:
#for Popularity:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
links = pd.read_csv('links.csv')
tags = pd.read_csv('tags.csv')
#with conversion to numeric all non convertibles will be replaced with NaN values
ratings['rating'] = pd.to_numeric(ratings['rating'], errors='coerce')

#we can drop the rows that contain NaN values in the rating column
ratings = ratings.dropna(subset=['rating'])

#the changes are then read back to our dataset csv file and ratings is read again
ratings.to_csv('ratings.csv', index=False)
ratings = pd.read_csv('ratings.csv')

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

ratingsAndMovies = ratings.merge(movies,on='movieId')

countRatingsDf = ratingsAndMovies.groupby('title').count()['rating'].reset_index()
countRatingsDf.rename(columns={'rating': 'numberOfRatings'}, inplace=True)

# Now, calculate the mean of 'rating' grouped by 'title'
averageRatingsDf = ratingsAndMovies.groupby('title')['rating'].mean().reset_index()
averageRatingsDf.rename(columns={'rating': 'averageOfRatings'}, inplace=True)

# Print the resulting DataFrame
print(averageRatingsDf)
popularityDf = countRatingsDf.merge(averageRatingsDf)

popularityDf = popularityDf[popularityDf['numberOfRatings'] >= 200]

# Lets sort such that the movies that have been rated the highest are on top:
popularityDf = popularityDf.sort_values('averageOfRatings', ascending=False)

popularityDf.merge(movies, on='title').drop_duplicates('title')

popularityDf.head(10)

In [ ]:
#for Collaborative:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
links = pd.read_csv('links.csv')
tags = pd.read_csv('tags.csv')

#with conversion to numeric all non convertibles will be replaced with NaN values
ratings['rating'] = pd.to_numeric(ratings['rating'], errors='coerce')

#we can drop the rows that contain NaN values in the rating column
ratings = ratings.dropna(subset=['rating'])

#the changes are then read back to our dataset csv file and ratings is read again
ratings.to_csv('ratings.csv', index=False)
ratings = pd.read_csv('ratings.csv')

# Split ratings data into training and testing sets
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

ratingsAndMovies = ratings.merge(movies,on='movieId')


x = ratingsAndMovies.groupby('userId').count()['rating'] > 100
#this returns true for the users that have and false for the ones that havent
filmGeeks = x[x].index
#filmGeeks will only store the users that have a rating > 100
userFilteredRatings = ratingsAndMovies[ratingsAndMovies['userId'].isin(filmGeeks)]
y = userFilteredRatings.groupby('title').count()['rating'] >= 50
#this returns a boolean set of values that we can store in y
# y[y] this stores only the boolean true returning values of y in y
highRatedMovies = y[y].index
userFilteredRatings['title'].isin(highRatedMovies)
#this will return boolean values
finalRatings = userFilteredRatings[userFilteredRatings['title'].isin(highRatedMovies)]

pt = finalRatings.pivot_table(index = 'title', columns = 'userId', values = 'rating')
pt.fillna(0,inplace=True)
similarityScores = cosine_similarity(pt)
# cosine_similarity(pt) returns 331 x 331 values
finalRatings.drop_duplicates('title')
def recommend(moviename):
    if moviename in pt.index:
        index = np.where(pt.index == moviename)[0][0]
        similarItems = sorted(list(enumerate(similarityScores[index])), key=lambda x: x[1], reverse=True)[1:6]
        for i in similarItems:
            print(pt.index[i[0]])
    else:
        print(f"{bookname} is not included in our high priority dataset for collaborative filtering.")
#checkValue = ratingsAndMovies[ratingsAndMovies['title'] == 'Saving Private Ryan (1998)'].head(1)
movieTitle = input("Enter the movie title: ")
recommend(movieTitle)

checkValue = ratingsAndMovies[ratingsAndMovies['title'] == 'Braveheart (1995)'].head(1)
print(checkValue)